In [9]:
%pip install PyPDF2 nltk groq

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import PyPDF2
import nltk
import re
from groq import Groq

nltk.download('stopwords')
nltk.download('punkt') # Required for word_tokenize
nltk.download('punkt_tab')
nltk.download('wordnet')
nltk.download('omw-1.4')

GROQ_API_KEY = ""
pdf_path = './file.pdf'

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\davi.soares\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\davi.soares\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\davi.soares\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\davi.soares\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\davi.soares\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [11]:
def extract_text_from_pdf(pdf_path):
    text = ""
    try:
        with open(pdf_path, 'rb') as file:
            reader = PyPDF2.PdfReader(file)
            for page in reader.pages:
                text += page.extract_text() + "\n"
    except FileNotFoundError:
        print(f"Error: The file '{pdf_path}' was not found.")
    except Exception as e:
        print(f"An error occurred while reading the PDF: {e}")
    return text

def clean_and_normalize_text(text):
    # Remove extra whitespaces, newlines, and form feeds
    text = re.sub(r'\s+', ' ', text)
    text = text.replace('\n', ' ').replace('\f', ' ')
    text = text.lower()
    text = text.strip()
    return text

raw_text = extract_text_from_pdf(pdf_path)
cleaned_text = clean_and_normalize_text(raw_text)

print("--- Cleaned and Normalized Text ---")
print(cleaned_text)
print("\n--- Total length of cleaned text ---")
print(len(cleaned_text))
print("\n--- Total length of raw text ---")
print(len(raw_text))

--- Cleaned and Normalized Text ---
distribution: initiated by: all departmental elements office of environment, safety and health approved: 3-28-00 nonreactor nuclear safety design criteria and explosives safety criteria guide for use with doe o 420.1, facility safety u.s. department of energy office of environment, safety and healthdoe g 420.1-1downloaded from https://www.everyspec.com doe g 420.1-1 i (and ii) 3-28-00 foreword this guide provides guidance on the application of requirements for nonreactor nuclear facilitiesand explosives facilities of department of energy (doe) o 420.1, facility safety,section 4.1, nuclear and explosives safety design criteria. the following guidelines wereestablished for the development of this guide.  this guide provides guidance on implementing the requirements stated in doe o 420.1, section 4.1, as they apply to the design aspects for nuclear safety of nonreactor nuclearfacilities and safety requirements for explosives facilities. the guidance pr

Now, let's update the preprocessing function to use lemmatization instead of stemming.

Now, let's remove punctuation, stop words, and apply stemming to the `cleaned_text`.

In [12]:
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

def preprocess_for_llm(text):
    text = re.sub(r'https?://\S+|www\.\S+', '', text)

    text = re.sub(r'[\W_]+', ' ', text)

    words = word_tokenize(text)

    stop_words = set(stopwords.words('english'))
    words = [word for word in words if word not in stop_words]

    lemmatizer = WordNetLemmatizer()
    words = [lemmatizer.lemmatize(word) for word in words]

    return ' '.join(words)

llm_prepared_text = preprocess_for_llm(cleaned_text)

print("--- Text Prepared for LLM Analysis (first 1000 characters) ---")
print(llm_prepared_text[:1000])
print("\n--- Total length of LLM-prepared text ---")
print(len(llm_prepared_text))

--- Text Prepared for LLM Analysis (first 1000 characters) ---
distribution initiated departmental element office environment safety health approved 3 28 00 nonreactor nuclear safety design criterion explosive safety criterion guide use doe 420 1 facility safety u department energy office environment safety healthdoe g 420 1 1downloaded doe g 420 1 1 ii 3 28 00 foreword guide provides guidance application requirement nonreactor nuclear facilitiesand explosive facility department energy doe 420 1 facility safety section 4 1 nuclear explosive safety design criterion following guideline wereestablished development guide guide provides guidance implementing requirement stated doe 420 1 section 4 1 apply design aspect nuclear safety nonreactor nuclearfacilities safety requirement explosive facility guidance provided thisguide restricted requirement identified doe 420 1 section 4 1 thisguide establish requirement safety analysis performed accordance doe std 3009 94 establish identification f

In [13]:
from collections import Counter
import re

text_without_numbers = re.sub(r'\d+', '', llm_prepared_text)

words_for_frequency = text_without_numbers.split()

words_for_frequency = [word for word in words_for_frequency if len(word) > 1]

word_counts = Counter(words_for_frequency)

print("\n--- 20 Most Frequent Words (Numbers and Single-Char Words Removed) ---")
for word, count in word_counts.most_common(20):
    print(f"{word}: {count}")


--- 20 Most Frequent Words (Numbers and Single-Char Words Removed) ---
safety: 414
design: 259
doe: 219
facility: 176
system: 170
standard: 131
ansi: 128
must: 127
requirement: 126
material: 109
nuclear: 106
sscs: 106
section: 101
american: 99
class: 85
analysis: 83
control: 83
national: 83
criterion: 80
ieee: 77


In [16]:
output_file_path = './llm_prepared_text.txt'

try:
    with open(output_file_path, 'w', encoding='utf-8') as f:
        f.write(llm_prepared_text)
    print(f"\n--- Successfully saved prepared text to '{output_file_path}' ---")
except Exception as e:
    print(f"An error occurred while saving the file: {e}")


--- Successfully saved prepared text to './llm_prepared_text.txt' ---


In [23]:
client = Groq(
    api_key=GROQ_API_KEY,
)
disciplines = ['Aerodinâmica', 'Elétrica', 'Ensaios Estruturais', 'Estruturas', 'Ferramental', 'Materiais e Processos', 'Pesos e Balanceamento', 'Sistemas Hidráulicos']

print("length of llm_prepared_text:", len(llm_prepared_text))

truncated_text = llm_prepared_text[:40000]

chat_completion = client.chat.completions.create(
    messages=[
        {
            "role": "system",
            "content": "You are an expert in engineering disciplines. Your task is to identify the most relevant engineering discipline from a given list based on a provided text. Respond only with the name of the discipline."
        },
        {
            "role": "user",
            "content": f"Analyze the following text:\n\n{truncated_text}\n\nBased on this text, which one of these disciplines is the most appropriate suggestion: {disciplines}? Provide only the name of the discipline, without any additional text or explanation.",
        }
    ],
    model="openai/gpt-oss-20b",
    temperature=0.7
)

print(chat_completion.choices[0].message.content)

length of llm_prepared_text: 108553


APIStatusError: Error code: 413 - {'error': {'message': 'Request too large for model `openai/gpt-oss-20b` in organization `org_01jxwbn306e5ct6dkvzv08c65p` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Requested 9205, please reduce your message size and try again. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}